<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Aranceles bilaterales y medidas no arancelarias

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.
- `glob` y `os`: para listar los archivos de medidas no arancelarias.
- `funciones.tariff_iso3`: conversión de nombres de país a códigos ISO3.

In [1]:
import pandas as pd
import numpy as np
import glob
import os
from funciones.tariff_iso3 import iso3

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import raw, temp

### Lectura de datos

Carga las tablas de correspondencia arancelaria preparadas previamente y deriva sus versiones
a cuatro dígitos, necesarias para los cruces de esta sección.

In [3]:
df_hs2012_2017 = pd.read_parquet(temp / 'df_hs2012_2017.parquet')
df_hs2012_2022 = pd.read_parquet(temp / 'df_hs2012_2022.parquet')

df_hs4_12_17 = df_hs2012_2017.copy()
df_hs4_12_17['HS4'] = df_hs4_12_17['HS6'].str[:4]
df_hs4_12_17['HS4_17'] = df_hs4_12_17['HS6_17'].str[:4]
df_hs4_12_17 = df_hs4_12_17.drop(columns=['HS6_17','HS6']).drop_duplicates()

### Aranceles bilaterales aplicados

Prepara la base de aranceles por país reportante, socio y producto.

**Pasos realizados:**

1. **Filtro por tipo de arancel:** conserva únicamente el arancel efectivamente aplicado (`AHS`).

2. **Estandarización del código:** completa el código de producto a seis dígitos.

3. **Cruce de nomenclatura:** traduce el código HS 2022 a HS 2012 y descarta los registros sin equivalencia.

4. **Agregación bilateral:** promedia el arancel ponderado y suma el valor importado por país y producto.

5. **Códigos ISO:** asigna el código ISO3 al país reportante y al socio.

In [4]:
df_aranceles = pd.read_parquet(raw / 'aranceles' / 'tariffs.parquet', columns=['Reporter Name','Product','Partner Name','Weighted Average','Imports Value in 1000 USD','DutyType']).query("DutyType == 'AHS'").drop(columns=['DutyType'])
df_aranceles['Product'] = df_aranceles['Product'].astype(str).str.zfill(6)
df_aranceles = df_aranceles.merge(df_hs2012_2022, how='left', left_on='Product', right_on='HS6_22').dropna(subset=['HS6']).groupby(['Reporter Name','Partner Name','HS6']).agg({'Weighted Average': 'mean','Imports Value in 1000 USD': 'sum'}).reset_index()
df_aranceles['Product'] = df_aranceles['HS6'].astype(int).astype(str).str.zfill(6)
df_aranceles['ISO Reporter'] = df_aranceles['Reporter Name'].apply(iso3)
df_aranceles['ISO Partner'] = df_aranceles['Partner Name'].apply(iso3)
df_aranceles = df_aranceles[['Reporter Name','ISO Reporter','Product','Partner Name','ISO Partner','Weighted Average','Imports Value in 1000 USD']]

temp.mkdir(parents=True, exist_ok=True)
df_aranceles.to_parquet(temp / 'df_aranceles.parquet')

### Aranceles anunciados por Estados Unidos

Lleva el arancel de proporción a porcentaje y fija los códigos ISO del país reportante y del socio.

In [5]:
df_trump = pd.read_excel(raw / 'aranceles' / 'Aranceles Trump.xlsx', sheet_name='Sheet2')
df_trump['Arancel'] = df_trump['Arancel'] * 100
df_trump['ISO Reporter'] = 'USA'
df_trump['ISO Partner'] = df_trump['ISO']

temp.mkdir(parents=True, exist_ok=True)
df_trump.to_parquet(temp / 'df_trump.parquet', index=False)

### Medidas no arancelarias

Consolida los archivos de medidas no arancelarias de la UNCTAD.

**Pasos realizados:**

1. **Carga:** lee todos los archivos de la carpeta omitiendo las seis filas de metadatos.

2. **Mercados afectados:** elimina las notas entre corchetes, separa los países listados en una misma celda y expande el resultado a una fila por país.

3. **Códigos arancelarios:** elimina las notas entre paréntesis, separa los códigos listados en una misma celda, los expande a una fila por código y conserva la porción numérica de al menos cuatro dígitos.

4. **Filtro y renombramiento:** descarta los registros dirigidos al mundo entero y renombra las columnas de país y código.

5. **Cruce de nomenclatura:** traduce el código a HS 2012 a seis dígitos y, cuando no hay correspondencia, a cuatro.

In [6]:
archivos = glob.glob(os.path.join(raw / 'aranceles' / 'ntms', '*.csv'))

df_ntms = pd.concat([pd.read_csv(archivo, header=6) for archivo in archivos], ignore_index=True)

df_ntms['Markets affected by NTM(s)'] = df_ntms['Markets affected by NTM(s)'].str.replace(r'\[.*?\]', '', regex=True)

df_ntms['Markets affected by NTM(s)'] = df_ntms['Markets affected by NTM(s)'].str.split(',')
df_ntms['Markets affected by NTM(s)'] = df_ntms['Markets affected by NTM(s)'].apply(lambda x: [word.strip() for word in x] if isinstance(x, list) else x)

df_ntms = df_ntms.explode('Markets affected by NTM(s)')

df_ntms['Hs code(s)'] = df_ntms['Hs code(s)'].str.replace(r'\(.*?\)', '', regex=True)

df_ntms['Hs code(s)'] = df_ntms['Hs code(s)'].str.split(',')
df_ntms['Hs code(s)'] = df_ntms['Hs code(s)'].apply(lambda x: [word.strip() for word in x] if isinstance(x, list) else x)

df_ntms = df_ntms.explode('Hs code(s)')

df_ntms = df_ntms[['Economy imposing NTM(s)', 'Markets affected by NTM(s)', 'NTM code', 'Hs code(s)']].query("`Markets affected by NTM(s)` != 'World'").rename(columns={'Economy imposing NTM(s)':'Reporter Name','Markets affected by NTM(s)':'Partner Name','Hs code(s)':'HS'})

df_ntms['HS'] = df_ntms['HS'].str.extract(r'(\d{4,})')[0]

df_ntms = df_ntms.merge(df_hs2012_2017, how='left', left_on='HS', right_on='HS6_17').merge(df_hs4_12_17, how='left', left_on='HS', right_on='HS4_17')
df_ntms['HS'] = df_ntms['HS6'].fillna(df_ntms['HS4'])
df_ntms = df_ntms.drop(columns=['HS6_17','HS4_17','HS6','HS4']).drop_duplicates().reset_index(drop=True)

temp.mkdir(parents=True, exist_ok=True)
df_ntms.to_parquet(temp / 'df_ntms.parquet')